[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/casos-de-uso/04-extraccion-pdfs.ipynb)

# 04 — Extracción de información de PDFs con IA

> **Bloque:** Casos de uso | **Nivel:** Práctico
>
> Complementario al tutorial [04-extraccion-pdfs.md](../../casos-de-uso/04-extraccion-pdfs.md)

Pipeline completo para leer PDFs y extraer datos estructurados con Claude.

In [ ]:
# %pip install anthropic pypdf python-dotenv pandas

import anthropic
import json
import os
from pathlib import Path

client = anthropic.Anthropic()
print('✓ Listo')

## 1. Funciones de extracción

In [ ]:
def leer_pdf(ruta: str) -> dict:
    """Extrae texto de un PDF."""
    import pypdf
    with open(ruta, 'rb') as f:
        reader = pypdf.PdfReader(f)
        paginas = [p.extract_text() or '' for p in reader.pages]
    texto = '\n\n'.join(paginas)
    return {
        'texto': texto,
        'num_paginas': len(paginas),
        'num_palabras': len(texto.split())
    }


def extraer_campos(texto: str, schema: dict, instrucciones: str = '') -> dict:
    """Extrae campos estructurados de un texto usando Claude."""
    prompt = f"""Extrae información del siguiente documento según el schema.
{instrucciones}

Schema (devuelve SOLO este JSON con los valores reales):
{json.dumps(schema, ensure_ascii=False, indent=2)}

Documento:
<documento>
{texto[:12000]}
</documento>"""

    r = client.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=1500,
        temperature=0.0,
        messages=[{'role': 'user', 'content': prompt}]
    )
    raw = r.content[0].text.strip()
    if raw.startswith('```'):
        raw = raw.split('```')[1]
        if raw.startswith('json'): raw = raw[4:]
    try:
        return json.loads(raw)
    except:
        return {'_error': 'JSON inválido', '_raw': raw}


print('✓ Funciones definidas')

## 2. Probar con texto simulado (sin PDF real)

In [ ]:
# Texto de factura simulada
texto_factura_simulado = """
FACTURA
Número: FAC-2026-0042
Fecha: 14 de abril de 2026
Vencimiento: 14 de mayo de 2026

EMISOR:
Tech Solutions S.L.
CIF: B12345678
Calle Mayor 42, 28001 Madrid
info@techsolutions.es

RECEPTOR:
Empresa Cliente S.A.
CIF: A87654321
Avenida de la Innovación 15, 08010 Barcelona

CONCEPTOS:
1. Desarrollo API de IA - 40 horas x 95€/h ............... 3.800,00€
2. Integración con Claude API - 20 horas x 95€/h ......... 1.900,00€
3. Licencia mensual plataforma ............................ 500,00€

Subtotal: 6.200,00€
IVA (21%): 1.302,00€
TOTAL: 7.502,00€

Forma de pago: Transferencia bancaria
IBAN: ES12 3456 7890 1234 5678 9012
"""

schema_factura = {
    'numero_factura': 'string',
    'fecha_emision': 'string (YYYY-MM-DD)',
    'fecha_vencimiento': 'string (YYYY-MM-DD)',
    'emisor': {'nombre': 'string', 'cif': 'string', 'email': 'string o null'},
    'receptor': {'nombre': 'string', 'cif': 'string'},
    'lineas': [{'descripcion': 'string', 'importe': 'number'}],
    'subtotal': 'number',
    'iva_porcentaje': 'number',
    'total': 'number',
    'iban': 'string o null'
}

print('Extrayendo datos de la factura...')
datos = extraer_campos(texto_factura_simulado, schema_factura,
                       'Extrae todos los datos financieros como números sin símbolo de moneda')
print(json.dumps(datos, ensure_ascii=False, indent=2))

## 3. Validar la coherencia de los datos extraídos

In [ ]:
def validar_factura(datos: dict) -> list:
    problemas = []
    if not datos.get('numero_factura'):
        problemas.append('⚠️ Número de factura no encontrado')
    if datos.get('subtotal') and datos.get('iva_importe') and datos.get('total'):
        calculado = datos['subtotal'] + datos.get('iva_importe', 0)
        if abs(calculado - datos['total']) > 0.02:
            problemas.append(f'⚠️ Total inconsistente: {calculado:.2f} ≠ {datos["total"]}')
    if datos.get('subtotal') and datos.get('iva_porcentaje'):
        iva_calc = round(datos['subtotal'] * datos['iva_porcentaje'] / 100, 2)
        total_calc = datos['subtotal'] + iva_calc
        if datos.get('total') and abs(total_calc - datos['total']) > 0.05:
            problemas.append(f'⚠️ Total no cuadra con subtotal x IVA: {total_calc:.2f} ≠ {datos["total"]}')
    return problemas

print('=== Validación ===')
problemas = validar_factura(datos)
if problemas:
    for p in problemas: print(p)
else:
    print('✅ Datos coherentes')

print(f'\nResumen:')
print(f'  Factura: {datos.get("numero_factura")}')
print(f'  Emisor:  {datos.get("emisor", {}).get("nombre")}')
print(f'  Total:   {datos.get("total")} €')

## 4. Procesar un PDF real (si tienes uno disponible)

In [ ]:
# Descomenta y modifica la ruta cuando tengas un PDF real

# RUTA_PDF = 'mi_factura.pdf'  # ← cambia por tu fichero

# if Path(RUTA_PDF).exists():
#     print(f'Leyendo {RUTA_PDF}...')
#     pdf = leer_pdf(RUTA_PDF)
#     print(f'Páginas: {pdf["num_paginas"]} | Palabras: {pdf["num_palabras"]:,}')
#
#     datos_reales = extraer_campos(pdf['texto'], schema_factura)
#     print(json.dumps(datos_reales, ensure_ascii=False, indent=2))
# else:
#     print(f'Fichero no encontrado: {RUTA_PDF}')

print('Celda lista — descomenta el código y proporciona un PDF real para probarla.')

## 5. Schema personalizado

Define tu propio schema para extraer cualquier tipo de información.

In [ ]:
# Texto de prueba — reemplaza por tu propio documento
mi_texto = """
ACTA DE REUNIÓN — 14 de abril de 2026

Asistentes: Ana García (CEO), Pedro López (CTO), María Ruiz (Product)

Temas tratados:
1. Revisión del roadmap de IA para Q2 2026
2. Aprobación del presupuesto para GPU cloud: 15.000€/mes
3. Contratación de 2 ingenieros ML para mayo

Decisiones:
- Se aprueba integrar Claude API en el producto principal
- Ana asignará presupuesto antes del 20 de abril
- Pedro liderará las entrevistas técnicas

Próxima reunión: 28 de abril de 2026
"""

mi_schema = {
    'fecha': 'string (YYYY-MM-DD)',
    'asistentes': ['lista de nombres y roles'],
    'temas': ['lista de temas tratados'],
    'decisiones': ['lista de decisiones adoptadas'],
    'tareas': [{'responsable': 'string', 'tarea': 'string', 'fecha': 'string o null'}],
    'proxima_reunion': 'string o null'
}

resultado = extraer_campos(mi_texto, mi_schema, 'Extrae todos los datos del acta de reunión')
print(json.dumps(resultado, ensure_ascii=False, indent=2))

---
**Tutorial relacionado:** [04-extraccion-pdfs.md](../../casos-de-uso/04-extraccion-pdfs.md)